<a href="https://colab.research.google.com/github/Hansini23/Statistical-Learning-e23291/blob/main/ME2050_Assignment_07b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Gaussian Mixture Clustering as Conditional Updating

## Setup (restated)

We have data $x_1,\dots,x_n\in\mathbb R^d$, a latent cluster label $C_i\in\{1,\dots,K\}$ with
$P(C_i=k)=\phi_k$, $\phi_k\ge 0$, $\sum_{k=1}^K\phi_k=1$, and

$$X_i\mid C_i=k \;\sim\; \mathscr N(\mu_k,\Sigma_k).$$

The parameters $\{\phi_k,\mu_k,\Sigma_k\}_{k=1}^K$ are fixed but unknown. Throughout, $\mathscr N(x\mid\mu,\Sigma)$ denotes the multivariate Gaussian density

$$\mathscr N(x\mid\mu,\Sigma)=\frac{1}{(2\pi)^{d/2}|\Sigma|^{1/2}}\exp\!\Big(-\tfrac12(x-\mu)^T\Sigma^{-1}(x-\mu)\Big).$$

---
## 1.

$C_i$ is a discrete random variable taking values in $\{1,\dots,K\}$, and $X_i$ is continuous conditional on $C_i$. The **law of total probability** (partitioning on the mutually exclusive, exhaustive events $\{C_i=k\}_{k=1}^K$) gives, for the joint density of $(X_i,C_i)$,

$$f_{X_i}(x_i)=\sum_{k=1}^K f_{X_i,C_i}(x_i,k)=\sum_{k=1}^K f_{X_i\mid C_i}(x_i\mid k)\,P(C_i=k).$$

By the model assumption $X_i\mid C_i=k\sim \mathscr N(\mu_k,\Sigma_k)$, we have $f_{X_i\mid C_i}(x_i\mid k)=\mathscr N(x_i\mid \mu_k,\Sigma_k)$, and $P(C_i=k)=\phi_k$. Substituting,

$$\boxed{\,p(x_i)=\sum_{k=1}^K \phi_k\, \mathscr N(x_i\mid \mu_k,\Sigma_k).\,}$$

**Why "Gaussian mixture" density.** The right-hand side is a convex combination of $K$ Gaussian densities: the weights $\phi_k$ are non-negative and sum to one, and each term $\mathscr N(x_i\mid\mu_k,\Sigma_k)$ is itself a valid density. A convex combination of densities is again a density (it integrates to 1 and is non‑negative), and it is called a **mixture density** because it is literally built by "mixing" — i.e. weighting and summing — several component densities together. Since every component here is Gaussian, $p(x_i)$ is called a *Gaussian mixture* density. Unlike a single Gaussian, it can be multimodal, skewed, or heavy-tailed, since it is a superposition of $K$ possibly very different bell curves.

---
## 2. Deriving the Posterior Cluster Probability

Bayes' rule applied to the discrete latent label $C_i$, conditioning on the observed continuous value $X_i=x_i$, gives

$$P(C_i=k\mid X_i=x_i)=\frac{P(X_i=x_i\mid C_i=k)\,P(C_i=k)}{P(X_i=x_i)}
=\frac{P(X_i=x_i\mid C_i=k)P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i\mid C_i=j)P(C_i=j)},$$

where the denominator is exactly the law-of-total-probability expansion from Part 1 (i.e. $p(x_i)$), written out over all $K$ competing clusters instead of collapsed into a single sum.

Substituting the Gaussian conditional density $P(X_i=x_i\mid C_i=k)=\mathscr N(x_i\mid\mu_k,\Sigma_k)$ and the prior $P(C_i=k)=\phi_k$ in both numerator and denominator:

$$\boxed{\,P(C_i=k\mid X_i=x_i)=\frac{\phi_k\,\mathscr N(x_i\mid \mu_k,\Sigma_k)}{\sum_{j=1}^K \phi_j\,\mathscr N(x_i\mid \mu_j,\Sigma_j)} \;=:\;\gamma_{ik}.\,}$$

**Why $\gamma_{ik}$ is a posterior probability of membership.** Before seeing $x_i$, our belief about cluster membership is the *prior* $\phi_k=P(C_i=k)$. Once $x_i$ is observed, Bayes' rule reweights this prior by how *compatible* $x_i$ is with each cluster's Gaussian shape, $\mathscr N(x_i\mid\mu_k,\Sigma_k)$ (the likelihood), and renormalizes by the total evidence $p(x_i)$ so the result sums to 1 over $k$. This is precisely conditional updating: $\gamma_{ik}$ is our updated (posterior) degree of belief that point $i$ came from cluster $k$, *given* that we actually observed $x_i$ — exactly analogous to updating a prior into a posterior anywhere else in Bayesian inference.

---
## 3. One-Hot Encoding of the Latent Cluster Variable

$Z_{ik}=\mathbb 1\{C_i=k\}$ is a Bernoulli random variable. Conditional on $X_i=x_i$, its expectation is, by definition of expectation of an indicator (a Bernoulli$(\gamma_{ik})$ variable given the data),

$$\mathbb E[Z_{ik}\mid X_i=x_i]=1\cdot P(Z_{ik}=1\mid X_i=x_i)+0\cdot P(Z_{ik}=0\mid X_i=x_i)=P(Z_{ik}=1\mid X_i=x_i).$$

But $\{Z_{ik}=1\}=\{C_i=k\}$ by definition of the one-hot encoding, so

$$\mathbb E[Z_{ik}\mid X_i=x_i]=P(C_i=k\mid X_i=x_i)=\gamma_{ik}.$$

Stacking this component-wise over $k=1,\dots,K$ (expectation of a random vector is applied entrywise),

$$\mathbb E[Z_i\mid X_i=x_i]=
\begin{bmatrix}
\mathbb E[Z_{i1}\mid X_i=x_i]\\
\vdots\\
\mathbb E[Z_{iK}\mid X_i=x_i]
\end{bmatrix}
=
\begin{bmatrix}\gamma_{i1}\\ \vdots \\ \gamma_{iK}\end{bmatrix}.$$

**Conclusion.** The vector of responsibilities is nothing but the conditional expectation of the (unobserved) one-hot cluster indicator given the observed data: $\mathbb E[Z_i\mid X_i=x_i]=(\gamma_{i1},\dots,\gamma_{iK})^T$. This is exactly what "soft cluster assignment" means — instead of committing to a single deterministic label, we describe $x_i$'s cluster membership by the *expected value* of its (unknown) one-hot label under the posterior distribution.

---
## 4. From Soft Assignment to Hard Clustering

$\mathbb E[Z_i\mid X_i=x_i]=(\gamma_{i1},\dots,\gamma_{iK})^T$ is a full probability vector over all $K$ clusters — it says, e.g., "this point is 70% cluster 1, 25% cluster 2, 5% cluster 3." This **soft assignment** preserves uncertainty: points near a cluster boundary get comparable responsibilities across two or more clusters, while points deep inside a cluster get a responsibility vector close to a one-hot vector.

**Hard clustering** collapses this whole distribution to a single label by taking the mode (MAP estimate),

$$\widehat C_i=\operatorname*{arg\,max}_{1\le k\le K}\gamma_{ik}.$$

This discards the magnitude of the uncertainty: two points, one with $\gamma_i=(0.99,0.01,0)$ and another with $\gamma_i=(0.34,0.33,0.33)$, would receive the *same* hard label if cluster 1 is the arg-max in both cases, even though the second point is essentially ambiguous between all three clusters. Soft clustering retains this distinction (useful for downstream tasks, uncertainty quantification, or as continuous weights in the M-step below); hard clustering is a convenient, interpretable summary obtained by thresholding the soft assignment at its maximum.

---
## 5. Conditional Expectation of the Observation Given the Cluster

By the model assumption, $X_i\mid C_i=k\sim\mathscr N(\mu_k,\Sigma_k)$. The mean parameter of a multivariate Gaussian *is*, by definition of the distribution, the expectation of a random vector following it:

$$\mathbb E[X_i\mid C_i=k]=\int x\,\mathscr N(x\mid \mu_k,\Sigma_k)\,dx=\mu_k.$$

**Interpretation of $\mu_k$.** $\mu_k$ is the point in $\mathbb R^d$ around which the mass of cluster $k$'s Gaussian is centered and symmetric; it is the value that minimizes the expected squared distance $\mathbb E\big[\lVert X_i-c\rVert^2 \mid C_i=k\big]$ over $c\in\mathbb R^d$. This is exactly the everyday sense of a cluster "center."

**Comparing the two conditional expectations.**

* $\mathbb E[Z_i\mid X_i=x_i]=(\gamma_{i1},\dots,\gamma_{iK})^T$ conditions **on the data point** $x_i$ and asks "which clusters is this point likely to belong to?" — its output lives in the probability simplex over cluster indices, and answers a *membership* question for one observation.
* $\mathbb E[X_i\mid C_i=k]=\mu_k$ conditions **on the cluster label** $k$ and asks "where in feature space does this cluster live?" — its output lives in $\mathbb R^d$, and answers a *location* question for one cluster.

The two expectations run in opposite directions of the same generative model: one infers the latent label from the observed data (posterior over $C_i$), the other describes the observed data generated by a fixed label (a property of the likelihood).

---
## 6. The Complete-Data Likelihood

If the true label $z_i$ (equivalently $c_i$) were known for every point, each factor $\big[\phi_k\mathscr N(x_i\mid\mu_k,\Sigma_k)\big]^{z_{ik}}$ equals $\phi_k\mathscr N(x_i\mid\mu_k,\Sigma_k)$ when $z_{ik}=1$ (i.e. $C_i=k$) and equals $1$ (any number to the power $0$) for every $j\ne k$. Hence the product over $k$ collapses to exactly one non-trivial factor — the density of the true generating cluster — for each $i$, and the product over $i$ gives the joint likelihood of all the (independent) data points and their labels:

$$p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_{i=1}^n\prod_{k=1}^K\Big[\phi_k\,\mathscr N(x_i\mid\mu_k,\Sigma_k)\Big]^{z_{ik}}.$$

Taking logarithms turns the outer product over $i$ into a sum, and $\log$ of a product over $k$ (with each term raised to the power $z_{ik}$) into a sum of $z_{ik}\log(\cdot)$ terms:

$$\ell_c=\log p(x_{1:n},z_{1:n})=\sum_{i=1}^n\sum_{k=1}^K z_{ik}\log\Big[\phi_k\,\mathscr N(x_i\mid\mu_k,\Sigma_k)\Big]
=\sum_{i=1}^n\sum_{k=1}^K z_{ik}\Big[\log\phi_k+\log\mathscr N(x_i\mid\mu_k,\Sigma_k)\Big].$$

**Why this is easy to maximize when $z_{ik}$ is known.** Because $z_{ik}\in\{0,1\}$ simply *selects* which cluster each point belongs to, $\ell_c$ decouples cleanly into $K$ independent, standard estimation problems: for each $k$, only the points with $z_{ik}=1$ contribute to the terms involving $\mu_k,\Sigma_k$, so maximizing over $(\mu_k,\Sigma_k)$ is just ordinary MLE of a multivariate Gaussian on the (now known) subset of points assigned to cluster $k$ — closed-form sample mean and sample covariance. Likewise the $\phi_k$'s only appear through $\sum_{i,k} z_{ik}\log\phi_k$, a Categorical/Multinomial log-likelihood whose closed-form MLE is the empirical cluster proportion. There is no coupling between clusters and no summation over an unknown label to marginalize — it is a fully-observed, separable estimation problem.

---
## 7. The EM Interpretation

In reality $Z_i$ is never observed, so $\ell_c$ cannot be computed or maximized directly. The EM algorithm instead works with the **expected** complete-data log-likelihood, taking the expectation of $\ell_c$ over the latent labels conditional on the observed data and the *current* parameter estimates $\theta^{\text{old}}=\{\phi_k^{\text{old}},\mu_k^{\text{old}},\Sigma_k^{\text{old}}\}$:

$$Q(\theta\mid\theta^{\text{old}})=\mathbb E_{Z\mid X,\theta^{\text{old}}}[\ell_c].$$

Because $\ell_c$ is **linear** in $z_{ik}$, expectation passes straight through the sum, and by Part 3, $\mathbb E[z_{ik}\mid x_i,\theta^{\text{old}}]=\gamma_{ik}$ (computed with $\theta^{\text{old}}$ plugged into the Gaussians and prior). So the unknown indicators are literally replaced by their conditional expectations, $z_{ik}\leadsto\gamma_{ik}$:

$$\boxed{\,Q=\sum_{i=1}^n\sum_{k=1}^K \gamma_{ik}\Big[\log\phi_k+\log\mathscr N(x_i\mid\mu_k,\Sigma_k)\Big].\,}$$

This is the **E-step**.

**Why the E-step is a conditional update.** Computing $\gamma_{ik}=P(C_i=k\mid X_i=x_i,\theta^{\text{old}})$ is exactly the Bayes'-rule computation of Part 2, carried out with the current (possibly still-inaccurate) parameter values. It re-estimates the posterior distribution over each point's cluster membership by conditioning on the observed data given the model's current state — a direct application of conditional updating, just repeated every iteration as the parameters $\theta$ themselves improve.

---
## 8. Parameter Updates

We maximize $Q$ separately over $\phi=(\phi_1,\dots,\phi_K)$, each $\mu_k$, and each $\Sigma_k$, holding $\gamma_{ik}$ fixed (they were computed in the E-step). Define $N_k=\sum_{i=1}^n\gamma_{ik}$ (the *effective* number of points softly assigned to cluster $k$). Note $\sum_k N_k=\sum_i\sum_k\gamma_{ik}=\sum_i 1=n$, since $\gamma_i$ is a probability vector for each $i$.

**(a) Mixture weights $\phi_k$.** Only the term $\sum_i\sum_k\gamma_{ik}\log\phi_k$ depends on $\phi$, subject to $\sum_k\phi_k=1$. Using a Lagrange multiplier $\lambda$:

$$\mathcal L=\sum_{i}\sum_k \gamma_{ik}\log\phi_k+\lambda\Big(\sum_k\phi_k-1\Big),
\qquad
\frac{\partial \mathcal L}{\partial\phi_k}=\frac{N_k}{\phi_k}+\lambda=0
\;\Rightarrow\;\phi_k=-\frac{N_k}{\lambda}.$$

Summing over $k$ and using $\sum_k\phi_k=1$, $\sum_k N_k=n$: $-\dfrac{n}{\lambda}=1\Rightarrow\lambda=-n$. Hence

$$\boxed{\phi_k^{\text{new}}=\frac{N_k}{n}.}$$

**(b) Means $\mu_k$.** Only $\log\mathscr N(x_i\mid\mu_k,\Sigma_k)=-\tfrac d2\log(2\pi)-\tfrac12\log|\Sigma_k|-\tfrac12(x_i-\mu_k)^T\Sigma_k^{-1}(x_i-\mu_k)$ depends on $\mu_k$. Differentiating $Q$ w.r.t. $\mu_k$ and setting it to zero:

$$\frac{\partial Q}{\partial \mu_k}=\sum_{i=1}^n\gamma_{ik}\,\Sigma_k^{-1}(x_i-\mu_k)=0
\;\Longrightarrow\;\sum_{i=1}^n\gamma_{ik}(x_i-\mu_k)=0
\;\Longrightarrow\;
\boxed{\mu_k^{\text{new}}=\frac{1}{N_k}\sum_{i=1}^n\gamma_{ik}x_i.}$$

**(c) Covariances $\Sigma_k$.** Writing the precision matrix $\Lambda_k=\Sigma_k^{-1}$ and using $\partial\log|\Lambda_k|/\partial\Lambda_k=\Sigma_k$ and $\partial\,(x-\mu_k)^T\Lambda_k(x-\mu_k)/\partial\Lambda_k=(x-\mu_k)(x-\mu_k)^T$,

$$\frac{\partial Q}{\partial \Lambda_k}=\sum_{i=1}^n\gamma_{ik}\Big[\tfrac12\Sigma_k-\tfrac12(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^T\Big]=0
\;\Longrightarrow\;
\boxed{\Sigma_k^{\text{new}}=\frac{1}{N_k}\sum_{i=1}^n\gamma_{ik}(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^T.}$$

This is the **M-step**: closed-form, weighted versions of the ordinary Categorical/Gaussian MLE formulas from Part 6, but with the unknown hard indicators $z_{ik}$ replaced everywhere by the soft responsibilities $\gamma_{ik}$.

**$\gamma_{ik}$ as a fractional membership weight.** In the updates above, point $i$ does not contribute a full, indivisible "vote" to exactly one cluster (as it would if $z_{ik}\in\{0,1\}$ were known); instead it contributes a *fraction* $\gamma_{ik}\in[0,1]$ of itself to cluster $k$'s weighted mean and weighted covariance, and the remaining fraction to the other clusters. $N_k=\sum_i\gamma_{ik}$ is therefore the *effective sample size* of cluster $k$ — a non-integer count of how many points, in expectation, belong to it — and each formula above is exactly a $\gamma_{ik}$-weighted average, reducing to the ordinary (hard) sample mean/covariance/proportion exactly when every $\gamma_{ik}\in\{0,1\}$.

---
## 9. Interpretation

Gaussian mixture clustering can be viewed as a repeated cycle of Bayesian conditional updating rather than a one-shot deterministic partitioning of the data. Before seeing any particular point, our belief about which cluster it came from is summarized by the prior $\phi_k=P(C_i=k)$. Once we observe $x_i$, the Gaussian density $\mathscr N(x_i\mid\mu_k,\Sigma_k)$ measures how compatible that specific location is with each cluster's shape, and Bayes' rule combines the prior and this compatibility into the responsibility $\gamma_{ik}$ — the posterior probability that point $i$ belongs to cluster $k$ *after* having observed $x_i$. Collecting these posterior probabilities across all $K$ clusters gives the soft assignment vector $\mathbb E[Z_i\mid X_i=x_i]=(\gamma_{i1},\dots,\gamma_{iK})^T$, a genuinely probabilistic (not deterministic) description of cluster membership. The M-step then closes the loop: it updates $\phi_k,\mu_k,\Sigma_k$ using these very responsibilities as weights, so the next round's "prior" and cluster shapes reflect what was just learned from the data. Iterating E- and M-steps is therefore nothing more than repeatedly forming a posterior over the latent labels given the current parameters, and then re-estimating the parameters given that posterior — i.e. **Gaussian mixture clustering is probabilistic clustering built entirely out of conditional expectations of a latent cluster-membership variable**, alternating between conditioning on data (E-step) and conditioning on labels (M-step) until convergence.

---
## 10. Computational Simulation and Out-of-Sample Validation

We now implement the theoretical framework above as a `GMMFinancialSegmenter` class, applied to the **Credit Card Dataset for Clustering** (Kaggle: [`arjunbhasin2013/ccdata`](https://www.kaggle.com/datasets/arjunbhasin2013/ccdata)), which summarizes the usage behavior of ~9,000 active credit card holders over 6 months (8,950 customers $\times$ 18 behavioral variables). Following the spirit of the assignment, we use two continuous behavioral features, `PURCHASES` (total purchase amount) and `CREDIT_LIMIT` (credit limit of the card), as the 2-D observations $x_i\in\mathbb R^2$ to be clustered.

**Data loading.** The cell below looks for `CC GENERAL.csv` locally first (e.g. if you downloaded it from Kaggle into the notebook's working directory, or are running on Kaggle itself under `/kaggle/input/ccdata/`); if it isn't found, it fetches a byte-identical public mirror of the same file over HTTPS so the notebook still runs end-to-end.

In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import plotly.express as px
import plotly.graph_objects as go

np.random.seed(42)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
def load_ccdata():
    # Locate CC GENERAL.csv locally, or download a mirror as a fallback.
    candidate_paths = [
        "CC GENERAL.csv",
        "CC_GENERAL.csv",
        "/kaggle/input/ccdata/CC GENERAL.csv",
        os.path.join(os.path.expanduser("~"), "CC GENERAL.csv"),
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            print(f"Loaded dataset from local file: {path}")
            return pd.read_csv(path)

    mirror_url = (
        "https://raw.githubusercontent.com/anurag-bhattacharjee/"
        "Credit-Card-Dataset/main/CC%20GENERAL.csv"
    )
    print("Local file not found — downloading dataset mirror ...")
    urllib.request.urlretrieve(mirror_url, "CC GENERAL.csv")
    print(f"Downloaded dataset to CC GENERAL.csv")
    return pd.read_csv("CC GENERAL.csv")


raw = load_ccdata()
print("Raw shape:", raw.shape)
raw[["PURCHASES", "CREDIT_LIMIT"]].describe()

Local file not found — downloading dataset mirror ...
Downloaded dataset to CC GENERAL.csv
Raw shape: (8950, 18)


,PURCHASES,CREDIT_LIMIT
count,"8,950.00","8,949.00"
mean,"1,003.20","4,494.45"
std,"2,136.63","3,638.82"
min,0.00,50.00
25%,39.63,"1,600.00"
50%,361.28,"3,000.00"
75%,"1,110.13","6,500.00"
max,"49,039.57","30,000.00"


### `GMMFinancialSegmenter`

The class encapsulates:

* **Data splitting & scaling** — drops missing rows on the chosen features, does an 80/20 train/test split, and standardizes (zero mean, unit variance) using statistics from the *training* set only (to avoid test-set leakage).
* **EM fitting** — fits a $K=3$ component `GaussianMixture` (scikit-learn's implementation of the EM algorithm derived in Parts 6–8) on the standardized training data, and reports convergence status and iteration count.
* **Out-of-sample validation** — `GaussianMixture.score` returns the average per-sample log-likelihood $\frac{1}{n_{\text{test}}}\sum_i \log p(x_i)$ (Part 1's marginal density, Part 8's fitted parameters) evaluated on held-out data, which is exactly the quantity we want to check for generalization of the learned density.
* **Responsibility contour** — evaluates $\gamma_{ik}$ (Part 2) on a fine grid across feature space and plots $\max_k \gamma_{ik}$ as a continuous background, which is the grid-wise version of the soft-assignment vector $\mathbb E[Z_i\mid X_i=x_i]$ from Part 3.

In [3]:
class GMMFinancialSegmenter:
    """
    Gaussian Mixture Model segmenter for 2-D financial behavior features,
    implementing the EM-based conditional-updating framework derived above.

    Parameters
    ----------
    n_components : int
        Number of Gaussian mixture components K.
    test_size : float
        Fraction of data held out for out-of-sample validation.
    random_state : int
        Seed for reproducibility (train/test split and EM initialization).
    """

    def __init__(self, n_components=3, test_size=0.2, random_state=42):
        self.n_components = n_components
        self.test_size = test_size
        self.random_state = random_state

        self.feature_cols = None
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(
            n_components=self.n_components,
            covariance_type="full",
            random_state=self.random_state,
            n_init=5,
        )

        self.X_train_raw = None
        self.X_test_raw = None
        self.X_train = None
        self.X_test = None

    # ------------------------------------------------------------------
    # Data preparation
    # ------------------------------------------------------------------
    def prepare_data(self, df, feature_cols=("PURCHASES", "CREDIT_LIMIT")):
        """Clean, split (80/20), and standardize the two chosen features."""
        self.feature_cols = list(feature_cols)

        data = df[self.feature_cols].dropna().copy()

        train_df, test_df = train_test_split(
            data, test_size=self.test_size, random_state=self.random_state
        )

        self.X_train_raw = train_df.values
        self.X_test_raw = test_df.values

        # Fit scaler on TRAIN ONLY to avoid test-set leakage
        self.X_train = self.scaler.fit_transform(self.X_train_raw)
        self.X_test = self.scaler.transform(self.X_test_raw)

        print(f"Train size: {self.X_train.shape[0]}  |  Test size: {self.X_test.shape[0]}")
        return self

    # ------------------------------------------------------------------
    # EM fitting
    # ------------------------------------------------------------------
    def fit(self):
        """Fit the GMM via EM on the standardized training data."""
        self.gmm.fit(self.X_train)

        status = "CONVERGED" if self.gmm.converged_ else "DID NOT CONVERGE"
        print(f"EM {status} after {self.gmm.n_iter_} iterations "
              f"(lower_bound_ = {self.gmm.lower_bound_:.4f})")
        return self

    # ------------------------------------------------------------------
    # Out-of-sample validation
    # ------------------------------------------------------------------
    def evaluate(self):
        """Average log-likelihood per sample on train vs held-out test set."""
        train_ll = self.gmm.score(self.X_train)   # mean log p(x_i), Part 1
        test_ll = self.gmm.score(self.X_test)
        print(f"Average log-likelihood  -> train: {train_ll:.4f}   test: {test_ll:.4f}")
        return train_ll, test_ll

    # ------------------------------------------------------------------
    # Grid of responsibilities gamma_ik  (Part 2 / Part 3)
    # ------------------------------------------------------------------
    def _responsibility_grid(self, x_range, y_range, n_grid=150):
        gx = np.linspace(*x_range, n_grid)
        gy = np.linspace(*y_range, n_grid)
        GX, GY = np.meshgrid(gx, gy)
        grid_raw = np.column_stack([GX.ravel(), GY.ravel()])
        grid_scaled = self.scaler.transform(grid_raw)

        # gamma_ik for every grid point and every component k
        gamma = self.gmm.predict_proba(grid_scaled)          # E[Z_i | X_i = x_grid]
        max_gamma = gamma.max(axis=1).reshape(GX.shape)       # max_k gamma_ik
        hard_label = gamma.argmax(axis=1).reshape(GX.shape)   # argmax_k gamma_ik
        return GX, GY, max_gamma, hard_label

    # ------------------------------------------------------------------
    # 1. Empirical density heatmap of the raw training data
    # ------------------------------------------------------------------
    def plot_density_heatmap(self):
        train_df = pd.DataFrame(self.X_train_raw, columns=self.feature_cols)
        fig = px.density_heatmap(
            train_df, x=self.feature_cols[0], y=self.feature_cols[1],
            marginal_x="histogram", marginal_y="histogram",
            nbinsx=60, nbinsy=60,
            title="Empirical 2D Density of Training Data (raw scale)",
        )
        fig.update_coloraxes(colorscale="Viridis")
        fig.update_layout(template="plotly_white", width=800, height=650)
        return fig

    # ------------------------------------------------------------------
    # 2 & 3. Assignment plots (train / test) with responsibility contour
    # ------------------------------------------------------------------
    def plot_assignment(self, split="train", n_grid=150, pad_frac=0.05):
        if split == "train":
            X_raw, title = self.X_train_raw, "Training Assignment"
        elif split == "test":
            X_raw, title = self.X_test_raw, "Test Assignment (out-of-sample)"
        else:
            raise ValueError("split must be 'train' or 'test'")

        x_min, x_max = self.X_train_raw[:, 0].min(), self.X_train_raw[:, 0].max()
        y_min, y_max = self.X_train_raw[:, 1].min(), self.X_train_raw[:, 1].max()
        x_pad = (x_max - x_min) * pad_frac
        y_pad = (y_max - y_min) * pad_frac
        x_range = (x_min - x_pad, x_max + x_pad)
        y_range = (y_min - y_pad, y_max + y_pad)

        GX, GY, max_gamma, hard_label = self._responsibility_grid(x_range, y_range, n_grid)

        # Hard labels of the plotted points, for coloring the scatter
        X_scaled = self.scaler.transform(X_raw)
        point_labels = self.gmm.predict(X_scaled)

        fig = go.Figure()

        fig.add_trace(go.Contour(
            x=GX[0], y=GY[:, 0], z=max_gamma,
            colorscale="Viridis", opacity=0.75,
            contours=dict(showlines=False),
            colorbar=dict(title="max<sub>k</sub> &gamma;<sub>ik</sub>"),
            name="max responsibility",
        ))

        for k in range(self.n_components):
            mask = point_labels == k
            fig.add_trace(go.Scatter(
                x=X_raw[mask, 0], y=X_raw[mask, 1],
                mode="markers",
                marker=dict(size=5, line=dict(width=0.3, color="white")),
                name=f"cluster {k}",
            ))

        fig.update_layout(
            title=f"{title}: points over posterior responsibility contour "
                  f"(K={self.n_components})",
            xaxis_title=self.feature_cols[0],
            yaxis_title=self.feature_cols[1],
            template="plotly_white", width=850, height=650,
        )
        return fig


### Running the pipeline

In [4]:
segmenter = GMMFinancialSegmenter(n_components=3, test_size=0.2, random_state=42)
segmenter.prepare_data(raw, feature_cols=("PURCHASES", "CREDIT_LIMIT"))
segmenter.fit()
train_ll, test_ll = segmenter.evaluate()

Train size: 7159  |  Test size: 1790
EM CONVERGED after 20 iterations (lower_bound_ = -1.6068)
Average log-likelihood  -> train: -1.6062   test: -1.6888


In [5]:
fig1 = segmenter.plot_density_heatmap()
fig1.show()

In [6]:
fig2 = segmenter.plot_assignment(split="train")
fig2.show()

In [7]:
fig3 = segmenter.plot_assignment(split="test")
fig3.show()

### Evaluation of the plots

**Density heatmap.** The raw `PURCHASES` vs `CREDIT_LIMIT` scatter is heavily right-skewed and concentrated near the origin, with a long, thinning tail toward customers with both high credit limits and high purchase volumes. The marginal histograms confirm each feature alone is unimodal but skewed rather than symmetric — a single 2-D Gaussian would fit this shape poorly, motivating a *mixture* of several Gaussians (some tight and centered on low-spend/low-limit customers, others broader and shifted toward high-spend/high-limit customers) to approximate the true multimodal, skewed density more flexibly, exactly as derived in Part 1.

**Training / test assignment plots.** The colored contour is $\max_k\gamma_{ik}$ evaluated over a fine grid spanning the feature space — i.e. it is the empirical realization of $\mathbb E[Z_i\mid X_i=x_{\text{grid}}]$ from Part 3, collapsed to its largest coordinate at every grid location. Where the contour is bright (close to 1), the model is confident: a point landing there would have a soft-assignment vector that is nearly one-hot, so hard and soft clustering agree. Where the contour dims toward $1/K$ (here, $1/3\approx0.33$), the grid location sits near the decision boundary between two or more Gaussians — a point there would receive comparable responsibility from multiple clusters, exposing genuine ambiguity in cluster membership rather than a forced deterministic choice. The scatter points are colored by their *hard* label $\widehat C_i=\arg\max_k \gamma_{ik}$ (Part 4), so points that sit visually close to a contour boundary but were assigned different colors are exactly the "hard call" cases whose soft assignment vector is far from one-hot.

Because the contour itself is computed **only from parameters fit on the training set**, and the test plot overlays genuinely held-out points on that same surface, the fact that test points continue to fall into high-confidence (bright) regions — rather than clustering along the low-confidence boundaries — is a visual counterpart to the out-of-sample log-likelihood score: both indicate the mixture learned on the training data generalizes to unseen customers rather than overfitting idiosyncrasies of the training split.